# Week 5 Lab: Scale and Route Numerical Features

Complete the cells marked **TODO**. Reuse the documented categorical behavior from Week 4 and add the required numerical scaling. Do not build a model in this lab.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

DATA_PATH = Path("data/week5_demo_accounts.csv")
accounts = pd.read_csv(DATA_PATH)
accounts.head()

## Part 1: Feature roles and original scale

The workflow specifies:

- Nominal: `region`, `contact_channel`
- Ordinal: `plan_tier`, `service_priority`
- Numerical: `monthly_charges`, `tenure_months`
- Exclude: `account_id`

The numerical branch must use `StandardScaler`.

In [ ]:
# TODO: Complete the three feature lists.
nominal_features = []
ordinal_features = []
numeric_features = []

plan_order = ["basic", "plus", "pro"]
priority_order = ["standard", "priority", "premium"]

feature_columns = nominal_features + ordinal_features + numeric_features
X = accounts[feature_columns]
X.head()

In [ ]:
X[numeric_features].agg(["min", "max", "mean", "std"])

**Before scaling:** Which numerical feature has the larger mean? Does that make it more important to a model? Explain.

TODO

## Part 2: Create and inspect the numerical scaler

In [ ]:
# TODO: Create the required numerical scaler.
numeric_scaler = None

In [ ]:
assert numeric_scaler is not None, "Create the scaler first."
numeric_array = numeric_scaler.fit_transform(X[numeric_features])
numeric_scaled = pd.DataFrame(numeric_array, columns=numeric_features)
numeric_scaled.head()

In [ ]:
pd.DataFrame(
    {
        "feature": numeric_features,
        "mean_": numeric_scaler.mean_,
        "scale_": numeric_scaler.scale_,
    }
)

Explain what a standardized value near `0`, above `0`, and below `0` means.

TODO

## Part 3: Combine all feature branches

In [ ]:
nominal_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

ordinal_encoder = OrdinalEncoder(
    categories=[plan_order, priority_order],
    handle_unknown="use_encoded_value",
    unknown_value=-1,
)

In [ ]:
# TODO: Build a ColumnTransformer with nominal, ordinal, and numeric branches.
# Use the supplied encoders and scaler. Enable verbose feature names.
preprocessor = None

In [ ]:
assert preprocessor is not None, "Complete the preprocessor first."
preprocessor.set_output(transform="pandas")
X_transformed = preprocessor.fit_transform(X)
assert isinstance(X_transformed, pd.DataFrame)
X_transformed.head()

In [ ]:
print("Original shape:", X.shape)
print("Transformed shape:", X_transformed.shape)
print(*preprocessor.get_feature_names_out(), sep="\n")

## Part 4: Transform new observations

In [ ]:
new_accounts = pd.DataFrame(
    {
        "region": ["east", "central"],
        "contact_channel": ["phone", "video"],
        "plan_tier": ["pro", "enterprise"],
        "service_priority": ["standard", "priority"],
        "monthly_charges": [110.00, 155.00],
        "tenure_months": [34, 1],
    }
)
new_accounts

In [ ]:
# TODO: Transform new_accounts with the already fitted preprocessor.
# Do not call fit or fit_transform.
new_transformed = None

In [ ]:
assert new_transformed is not None, "Transform the new observations first."
assert isinstance(new_transformed, pd.DataFrame)
new_transformed

## Part 5: Interpret the result

1. What do the signs of the standardized numerical values tell you?
2. Where did the scaler's mean and scale come from?
3. Why did you call `transform()` instead of `fit_transform()` on `new_accounts`?
4. Why can a new standardized value be much greater than `1`?
5. How do the unseen categorical values behave?

TODO

## Part 6: Verify

In [ ]:
# TODO: Determine the expected transformed column count.
expected_columns = None

assert X_transformed.shape == (len(accounts), expected_columns)
assert new_transformed.shape == (2, expected_columns)

numeric_columns = [f"numeric__{name}" for name in numeric_features]
assert np.allclose(X_transformed[numeric_columns].mean(), 0.0, atol=1e-12)
assert np.allclose(X_transformed[numeric_columns].std(ddof=0), 1.0, atol=1e-12)

fitted_scaler = preprocessor.named_transformers_["numeric"]
expected_monthly = (
    new_accounts.loc[0, "monthly_charges"] - fitted_scaler.mean_[0]
) / fitted_scaler.scale_[0]
assert np.isclose(new_transformed.loc[0, "numeric__monthly_charges"], expected_monthly)

print("All lab checks passed.")

## Reflection

- What made sense this week?
- What was difficult or unclear?
- What do you still have questions about?
- What would you check first if the numerical results looked wrong for new data?

TODO